In [6]:
import pandas as pd

dfs = [
    pd.read_csv("tweets.csv"),
    pd.read_csv("splitting/instance_1/tweets_1.csv"),
    pd.read_csv("splitting/instance_2/tweets_2.csv"),
    pd.read_csv("splitting/instance_3/tweets_3.csv"),
]

merged = pd.concat(dfs, ignore_index=True)

# Drop any duplicates in case of overlap
merged = merged.drop_duplicates(subset=["Tweet_ID"])

print(f"Total tweets: {len(merged):,}")
merged.to_csv("tweets_all.csv", index=False)

Total tweets: 498,490


In [7]:

counts = merged.groupby("Account").size().reset_index(name="tweet_count")
counts = counts.sort_values("tweet_count", ascending=False)
print(f"\nTweets per account:")
print(counts.to_string())
print(f"\nTotal accounts: {counts['Account'].nunique()}")


Tweets per account:
             Account  tweet_count
396        mtgreenee        13298
394          ewarren        12904
3      BernieSanders        12792
0                AOC         9993
379       TeamPelosi         6429
23      EliCrane_CEO         4166
135      RepDonBacon         3987
251     RepMarkGreen         3528
149     RepEspaillat         3398
340  RepThomasMassie         3329
26      FrankPallone         3150
151      RepFeenstra         3145
196       RepJayapal         3069
328      RepStefanik         3032
389   WarrenDavidson         3024
360      RepYoungKim         2994
214         RepJoshG         2993
373        SenWarren         2985
206  RepJimmyPanetta         2961
393      debbielesko         2836
97        RepChipRoy         2804
55    RepAndyBiggsAZ         2768
326    RepSpanberger         2691
338        RepTenney         2656
271   RepMikeQuigley         2655
138  RepDonaldsPress         2632
269    RepMikeLawler         2600
84      RepBrianFitz       

In [4]:

# Accounts in sample with no tweets at all
sample = pd.read_csv("sample_house_full.csv")
sample["handle_lower"] = sample["twitter"].str.lower().str.strip()
scraped = set(merged["Account"].str.lower().str.strip())
no_tweets = sample[~sample["handle_lower"].isin(scraped)]
print(f"\nAccounts with no tweets scraped: {len(no_tweets)}")
print(no_tweets[["official_full", "twitter"]].to_string())


Accounts with no tweets scraped: 22
                 official_full          twitter
30                 Wesley Bell  RepWesleyBellMO
42                Mike Johnson   RepMikeJohnson
46                 Wesley Hunt    RepWesleyHunt
49               Michael Waltz  RepMichaelWaltz
69              Michelle Steel         RepSteel
82        Christopher H. Smith    RepChrisSmith
91            Craig A. Goldman  RepCraigGoldman
104             Mikie Sherrill      RepSherrill
107                 Bill Posey    CongBillPosey
115  Matthew M. Rosendale, Sr.     RepRosendale
116          Alan S. Lowenthal     RepLowenthal
125         Nicole Malliotakis   RepMalliotakis
131              Michael Cloud       RepCloudTX
196                  Tom Emmer      RepTomEmmer
262         Blaine Luetkemeyer   RepBlainePress
263         Katherine M. Clark        RepKClark
310              Anna G. Eshoo     RepAnnaEshoo
318            James C. Moylan   JMoylanforGuam
364            Earl Blumenauer  BlumenauerMedia
387

In [5]:
# Accounts with tweets but handle changes

# Handle replacements
replacements = {
    "RepKClark":       "WhipKClark",
    "RepMikeJohnson":  "SpeakerJohnson",
    "RepWesleyHunt":   "RepWPH",
    "RepMalliotakis":  "RepNicole",
    "RepCloudTX":      "RepMichaelCloud",
    "BlumenauerMedia": "repblumenauer",
    "RepBlainePress":  "RepBlaine",
    "RepLanceGooden":  "Lancegooden",
    "JMoylanforGuam": "RepMoylan",  # same, no change needed
    "RepTomEmmer":     "GOPMajorityWhip",
    "RepRosendale":    "FmrRepRosendale",
}

sample = pd.read_csv("sample_house_full.csv")

# Apply replacements
sample["twitter"] = sample["twitter"].replace(replacements)
sample.to_csv("sample_house_full.csv", index=False)
print("Updated sample_house_full.csv")

# Export just the new handles as a rescrape list
rescrape = pd.DataFrame({
    "twitter": [v for k, v in replacements.items() if k != v]
})
rescrape.to_csv("rescrape_handles.csv", index=False)
print(f"\nHandles to rescrape ({len(rescrape)}):")
print(rescrape["twitter"].tolist())

Updated sample_house_full.csv

Handles to rescrape (11):
['WhipKClark', 'SpeakerJohnson', 'RepWPH', 'RepNicole', 'RepMichaelCloud', 'repblumenauer', 'RepBlaine', 'Lancegooden', 'RepMoylan', 'GOPMajorityWhip', 'FmrRepRosendale']


In [9]:

tweets = pd.read_csv("tweets_all.csv")
tweets["Created At"] = pd.to_datetime(tweets["Created At"], errors="coerce", utc=True)

start = pd.Timestamp("2023-01-01", tz="UTC")
end   = pd.Timestamp("2024-11-30", tz="UTC")

filtered = tweets[(tweets["Created At"] >= start) & (tweets["Created At"] <= end)]

print(f"Before filter: {len(tweets):,}")
print(f"After filter : {len(filtered):,}")
print(f"Dropped      : {len(tweets) - len(filtered):,}")

filtered.to_csv("tweets_filtered.csv", index=False)

/var/folders/st/yx5_mdvj4bdbm59sg4_7n26m0000gq/T/ipykernel_30869/334555762.py:1: DtypeWarning: Columns (1,3,7,8) have mixed types. Specify dtype option on import or set low_memory=False.
  tweets = pd.read_csv("tweets_all.csv")
/var/folders/st/yx5_mdvj4bdbm59sg4_7n26m0000gq/T/ipykernel_30869/334555762.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  tweets["Created At"] = pd.to_datetime(tweets["Created At"], errors="coerce", utc=True)


Before filter: 498,490
After filter : 241,253
Dropped      : 257,237


In [10]:
# Check date range of what was dropped
dropped = tweets[~tweets.index.isin(filtered.index)]
print("Dropped date range:")
print(dropped["Created At"].min())
print(dropped["Created At"].max())
print()
print("NaT count:", dropped["Created At"].isna().sum())
print("Pre-2023:", (dropped["Created At"] < start).sum())
print("Post-2024:", (dropped["Created At"] > end).sum())

Dropped date range:
2016-01-01 00:54:03+00:00
2026-02-08 23:08:18+00:00

NaT count: 197421
Pre-2023: 53705
Post-2024: 6111


In [11]:
raw = pd.read_csv("tweets_all.csv", low_memory=False)
bad = raw[pd.to_datetime(raw["Created At"], errors="coerce", utc=True).isna()]
print(bad["Created At"].value_counts().head(20))

/var/folders/st/yx5_mdvj4bdbm59sg4_7n26m0000gq/T/ipykernel_30869/1407253115.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  bad = raw[pd.to_datetime(raw["Created At"], errors="coerce", utc=True).isna()]


Series([], Name: count, dtype: int64)


In [ ]:
raw = pd.read_csv("tweets_all.csv", low_memory=False)
print(raw["Created At"].isna().sum())
print(raw[raw["Created At"].isna()].head(10))